In [1]:
import json, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
sys.path.insert(0, str(Path.cwd().parent / 'python'))
from analysis_common import analytical_src, segment_metrics
from config import AGGREGATE_DIR, AUDIT_DIR, CLEAN_PARQUET, DOCS_DIR, STATS_DIR, VISUALS_DIR, connect
pd.set_option('display.max_columns', 40, 'display.width', 180, 'display.float_format', '{:,.4f}'.format)
con = connect(); src = analytical_src()
kpi = json.loads((AGGREGATE_DIR / 'portfolio_kpis.json').read_text())
fin = json.loads((AGGREGATE_DIR / 'financial_totals.json').read_text())
show = lambda name: display(Image(filename=str(VISUALS_DIR / f'{name}.png')))
csv = lambda name: pd.read_csv(AGGREGATE_DIR / f'{name}.csv')

# 7. Risk Segmentation
Transparent, retrospective tiers (`python/risk_segmentation.py`).

Relative Risk Index = segment mature-cohort rate / portfolio mature-cohort rate. Low: under 0.75. Medium: 0.75 to under 1.25. High: 1.25 or more. Fewer than 500 mature resolved loans: Insufficient data.

These classify past portfolio behaviour. They are not underwriting rules or individual credit decisions, and they use no protected characteristics.

In [2]:
csv('risk_segments_grade_term')[['risk_segment','loans','mature_resolved','default_rate','mature_default_rate','relative_risk_index','risk_tier','share_of_funded','share_of_unpaid_principal']]

,risk_segment,loans,mature_resolved,default_rate,mature_default_rate,relative_risk_index,risk_tier,share_of_funded,share_of_unpaid_principal
0,A-36m,409181,"165,804.0000",0.0592,0.0549,0.3625,Low,0.1705,0.0351
1,A-60m,23846,"1,215.0000",0.1061,0.0831,0.5487,Low,0.0154,0.0032
2,B-36m,534906,"242,619.0000",0.1274,0.1141,0.7531,Medium,0.1949,0.1086
3,B-60m,128651,"9,099.0000",0.1801,0.1528,1.0083,Medium,0.0816,0.0429
4,C-36m,419858,"184,598.0000",0.2049,0.1834,1.2105,Medium,0.1492,0.1503
5,C-60m,230195,"22,242.0000",0.2753,0.2152,1.4206,High,0.1382,0.1440
6,D-36m,184199,"84,939.0000",0.2689,0.2400,1.5842,High,0.0668,0.1017
7,D-60m,140225,"14,825.0000",0.3627,0.2721,1.7960,High,0.0831,0.1392
8,E-36m,50791,"24,535.0000",0.3269,0.2978,1.9657,High,0.0181,0.0375
9,E-60m,84848,"12,385.0000",0.4210,0.3176,2.0960,High,0.0515,0.1329


In [3]:
csv('risk_tier_summary')

,risk_tier,loans,funded_amount,resolved,charged_off,unpaid_principal_at_chargeoff,default_rate,share_of_funded,share_of_unpaid_principal
0,High,744226,"13,207,648,400.0000","441,365.0000","145,205.0000","1,979,370,919.4700",0.3290,0.3884,0.6599
1,Low,433027,"6,321,329,325.0000","235,090.0000","14,201.0000","114,894,432.6600",0.0604,0.1859,0.0383
2,Medium,1083415,"14,475,230,875.0000","668,855.0000","109,153.0000","905,319,075.0200",0.1632,0.4257,0.3018


In [4]:
s = csv('risk_segment_summary'); s[s.risk_tier=='High'].sort_values('relative_risk_index', ascending=False)[['dimension','segment','mature_resolved','mature_default_rate','relative_risk_index','share_of_funded','share_of_unpaid_principal']].head(20)

,dimension,segment,mature_resolved,mature_default_rate,relative_risk_index,share_of_funded,share_of_unpaid_principal
36,sub_grade,F5,"1,304.0000",0.4095,2.7029,0.0031,0.0110
84,rate_band,25%+,"2,407.0000",0.4055,2.6764,0.0270,0.0725
37,sub_grade,G1,874.0000,0.3993,2.6356,0.0025,0.0082
6,grade,G,"2,350.0000",0.3902,2.5756,0.0073,0.0267
38,sub_grade,G2,630.0000,0.3889,2.5668,0.0016,0.0060
35,sub_grade,F4,"1,732.0000",0.3695,2.4390,0.0035,0.0128
5,grade,F,"11,665.0000",0.3525,2.3267,0.0235,0.0779
34,sub_grade,F3,"2,369.0000",0.3520,2.3237,0.0043,0.0146
33,sub_grade,F2,"2,629.0000",0.3518,2.3223,0.0052,0.0173
31,sub_grade,E5,"4,720.0000",0.3307,2.1829,0.0122,0.0309
